# Track B — one match, start to finish (Colab)

Run the cells top to bottom for **one match**. Everything lands in Google Drive under `MyDrive/football-analysis`, and every step can be re-run: after a disconnect or a "GPU limit" message, reconnect and run the cells again — finished 10-minute chunks are skipped.

**Before you start**
- *Steps 1–3 (setup, details, download)* need **no GPU**. Run them on a normal CPU runtime to save compute units.
- *Steps 4–5 (smoke test, full run)*: **Runtime → Change runtime type → T4 GPU** (or L4 if offered). Turn on **High-RAM** if your plan shows the toggle.
- Keep this browser tab open while step 5 runs (about 1–1.5 h per match on a T4).

The notebook kernel never loads PyTorch; each chunk runs as its own process, so RAM and GPU memory are released every 10 minutes of video.

In [ ]:
# @title 0 · Choose the match
MATCH_ID = "persiba_persika_20261121"  # @param ["persiba_persika_20261121", "persiba_gresik_20261113", "persipa_persiba_20261117", "persika_persinga_20261113", "gresik_persika_20261117", "persinga_psf_20261118", "persinga_persibangga_20261122"] {allow-input: true}
ROOT = "/content/drive/MyDrive/football-analysis"  # @param {type:"string"}
WORK = "/content/work"   # fast local disk; wiped when the runtime resets (that's fine)
BRANCH = "main"
READ_NUMBERS = False   # True = also read shirt numbers (OCR; pre-fills player numbers in review)
CONFIG_NAME = "persiba_2026-27.json"
print("Match:", MATCH_ID)

In [ ]:
# @title 1 · Setup: Drive, code, packages (about 2–3 min)
import os, subprocess, sys, shutil, json
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

REPO = Path("/content/football-analytics")
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/mryudistira21/football-analytics.git", str(REPO)], check=True)
print("Code:", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

# Colab already has a CUDA build of PyTorch; requirements.txt does not touch it.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt"),
                "git+https://github.com/roboflow/sports.git", "psutil",
                *(["easyocr"] if READ_NUMBERS else [])], check=True)
if shutil.which("deno") is None:   # JavaScript runtime yt-dlp uses for YouTube
    subprocess.run("curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1", shell=True)
    os.environ["PATH"] = str(Path.home() / ".deno" / "bin") + os.pathsep + os.environ["PATH"]

# The match config lives in Drive so your edits survive runtime resets.
CFG = Path(ROOT) / "matches" / CONFIG_NAME
CFG.parent.mkdir(parents=True, exist_ok=True)
if not CFG.exists():
    shutil.copy(REPO / "matches" / CONFIG_NAME, CFG)
    print("Config copied to", CFG)
else:
    # add matches that exist in the repo but not yet in your Drive copy
    mine, repo = json.loads(CFG.read_text()), json.loads((REPO / "matches" / CONFIG_NAME).read_text())
    have = {m["match_id"] for m in mine["matches"]}
    added = [m for m in repo["matches"] if m["match_id"] not in have]
    new_defaults = {k: v for k, v in repo.get("defaults", {}).items() if k not in mine.get("defaults", {})}
    if added or new_defaults:
        mine["matches"] += added
        mine.setdefault("defaults", {}).update(new_defaults)
        CFG.write_text(json.dumps(mine, indent=2))
        print("Added to your config:", [m["match_id"] for m in added], list(new_defaults))
print("Config:", CFG)
os.chdir(REPO)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU on this runtime (fine for steps 1-3)"
!free -h | head -2

In [ ]:
# @title 2 · Match details — leave a field blank to keep what is in the config
VIDEO_URL = ""     # @param {type:"string"}
P1_START = ""      # @param {type:"string"}
P1_END = ""        # @param {type:"string"}
P2_START = ""      # @param {type:"string"}
P2_END = ""        # @param {type:"string"}
HOME_DIR_1H = "keep"   # @param ["keep", "R", "L"]
SWAP_TEAMS = "keep"    # @param ["keep", "no", "yes"]
# Times = elapsed time in the downloaded video (MM:SS or H:MM:SS), real kick-off -> real final
# whistle of each half, stoppage time INCLUDED. No extra time in league matches.
# HOME_DIR_1H: R if the HOME team attacks left->right on screen in the 1st half, else L.
# SWAP_TEAMS: "yes" if the smoke-test stills show the home team in ORANGE.
import json
cfg = json.loads(CFG.read_text())
m = next(x for x in cfg["matches"] if x["match_id"] == MATCH_ID)
if VIDEO_URL.strip(): m["video_url"] = VIDEO_URL.strip()
per = m.setdefault("periods", {})
for p, (a, b) in {"1": (P1_START, P1_END), "2": (P2_START, P2_END)}.items():
    cur = per.get(p) or [None, None]
    per[p] = [a.strip() or cur[0], b.strip() or cur[1]]
    if per[p] == [None, None]: per[p] = None
if HOME_DIR_1H != "keep": m["home_dir_1h"] = HOME_DIR_1H
if SWAP_TEAMS != "keep": m["swap_teams"] = SWAP_TEAMS == "yes"
CFG.write_text(json.dumps(cfg, indent=2))
print(json.dumps(m, indent=2))

In [ ]:
# @title 3 · Download the match video to Drive (CPU is fine; skipped if already there)
!python scripts/run_match.py --config "$CFG" --match "$MATCH_ID" --root "$ROOT" --work "$WORK" --download-only

**Now find the period times.** Open the video from Drive (`football-analysis/videos/`) or on YouTube, note the elapsed video time of each kick-off and each final whistle (stoppage time included), and enter them in step 2. Also note which way the home team attacks in the 1st half. Switch to a **GPU runtime** for steps 4–5 (re-run 0–2 after switching; the video stays in Drive).

In [ ]:
# @title 4 · Smoke test: 60 s with team colours (≈3–5 min incl. model download)
!python scripts/run_match.py --config "$CFG" --match "$MATCH_ID" --root "$ROOT" --work "$WORK" --smoke
from IPython.display import Image, display
from pathlib import Path
for p in sorted((Path(ROOT) / "runs" / MATCH_ID / "smoke").glob("still_*.jpg")):
    display(Image(filename=str(p), width=900))

**Check the stills:** GREEN boxes are labelled *home*, ORANGE *away*, grey = unknown/referee.
- Home team in **orange**? → step 2: `SWAP_TEAMS = yes`, run step 2. No rerun of tracking needed.
- Mostly grey after a minute is normal at the start; it fills in as tracks get classified.
- To refit the team colours (e.g. the first minutes were replays), delete `runs/<match>/team-state.joblib` in Drive and run step 4 again.

In [ ]:
# @title 5 · Full match — resumable. Re-run this cell after any disconnect.
!python scripts/run_match.py --config "$CFG" --match "$MATCH_ID" --root "$ROOT" --work "$WORK" --no-db {"--read-numbers" if READ_NUMBERS else ""}

In [ ]:
# @title 6 · Results: report preview + download the bundle
from pathlib import Path
from IPython.display import Markdown, display
run = Path(ROOT) / "runs" / MATCH_ID
md_file = run / "report" / f"{MATCH_ID}_team_report.md"
if md_file.exists():
    display(Markdown(md_file.read_text()))
else:
    print("No report yet: set HOME_DIR_1H in step 2, then run step 7.")
bundle = run / f"{MATCH_ID}_trackb_bundle.zip"
print("Bundle:", bundle, f"({bundle.stat().st_size/2**20:.1f} MB)" if bundle.exists() else "(not built yet)")
from google.colab import files
if bundle.exists():
    files.download(str(bundle))

In [ ]:
# @title 7 · (No GPU) rebuild report/moments after changing HOME_DIR_1H or SWAP_TEAMS
!python scripts/run_match.py --config "$CFG" --match "$MATCH_ID" --root "$ROOT" --work "$WORK" --post-only --no-db

In [ ]:
# @title 8 · (Optional) half-by-half review clips — stream copy, seconds not hours
# Cuts at the nearest keyframe (~1-2 s early), no re-encode. Clips are for watching only:
# keep tagging and Track B on the ORIGINAL master so video_seconds stay shared.
import json, subprocess
from pathlib import Path
cfg = json.loads(CFG.read_text()); m = next(x for x in cfg["matches"] if x["match_id"] == MATCH_ID)
def secs(t):
    p = [float(x) for x in str(t).split(":")]
    return p[0]*60+p[1] if len(p) == 2 else p[0]*3600+p[1]*60+p[2]
master = next(p for p in sorted((Path(ROOT) / "videos").glob(MATCH_ID + ".*")) if p.suffix in (".mp4", ".mkv", ".webm"))
out_dir = Path(ROOT) / "videos" / "clips"; out_dir.mkdir(exist_ok=True)
for p, rng in sorted((m.get("periods") or {}).items()):
    if not rng: continue
    s, e = secs(rng[0]), secs(rng[1])
    out = out_dir / f"{MATCH_ID}_half{p}.mp4"
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-ss", f"{max(0, s - 2):.2f}",
                    "-i", str(master), "-t", f"{e - s + 4:.2f}", "-c", "copy", "-avoid_negative_ts", "make_zero",
                    str(out)], check=True)
    print("saved", out)